# YOLOv8-pose 模型解读

> 当前打开的文件：`ultralytics-8.4.113/ultralytics/cfg/models/v8/yolov8-pose.yaml`

## 1. 一句话概述

**YOLOv8-pose = YOLOv8 目标检测 + 关键点估计（Pose）**。它在一张图上同时完成三件事：

1. **检测**：找出目标（人）的边界框 `[x1,y1,x2,y2]`
2. **分类**：判断目标类别（COCO 姿势数据集中就是 `person` 1 类）
3. **关键点**：回归 17 个关键点坐标 + 可见性，如 `nose, eyes, ears, shoulders, elbows, wrists, hips, knees, ankles`

一次前向即可输出：`框 + 类别 + 关键点`，无需额外后处理网络。

**整体架构（三段式）**

```
输入 640×640
   │
   ▼
Backbone (CSPDarknet 风格: Conv + C2f + SPPF) ──► P3/8, P4/16, P5/32 三个尺度特征
   │
   ▼
Neck (FPN + PAN 特征金字塔) ──► 多尺度特征融合
   │
   ▼
Pose Head (框分支 cv2 + 类分支 cv3 + 关键点分支 cv4)
   │
   ▼
输出: [x1,y1,x2,y2, conf, cls, 17×3 关键点]
```

下面按 配置文件 → 主干 → Neck → Pose 头 → 解码 → 应用 的顺序逐层解读。


## 2. 配置文件 yolov8-pose.yaml 解读

**头部参数**
```yaml
nc: 1                 # 类别数 (COCO 关键点数据集只有 person 1 类)
kpt_shape: [17, 3]    # 17 个关键点 × 3 维 (x, y, visible)
scales:               # 复合缩放系数 [depth, width, max_channels]
  n: [0.33, 0.25, 1024]   # nano
  s: [0.33, 0.50, 1024]   # small
  m: [0.67, 0.75, 768]    # medium
  l: [1.00, 1.00, 512]    # large
  x: [1.00, 1.25, 512]    # xlarge
```

- `kpt_shape[1]=2` 表示只预测 x,y；`=3` 额外预测可见性 visible。
- `scales` 是**复合缩放**：`yolov8n-pose.yaml` 实际就是调用本文件并把 `n` 的参数替换进去（nano 版把层深×0.33、通道宽×0.25）。

**Backbone 主干（8 层 + SPPF）**
| 层 | 模块 | 参数 | 输出 |
|---|---|---|---|
| 0 | Conv 3x3 s2 | 64 | P1/2 |
| 1 | Conv 3x3 s2 | 128 | P2/4 |
| 2 | C2f ×3 | 128 | |
| 3 | Conv 3x3 s2 | 256 | **P3/8** |
| 4 | C2f ×6 | 256 | |
| 5 | Conv 3x3 s2 | 512 | **P4/16** |
| 6 | C2f ×6 | 512 | |
| 7 | Conv 3x3 s2 | 1024 | **P5/32** |
| 8 | C2f ×3 | 1024 | |
| 9 | SPPF | 1024, k=5 | 多尺度池化融合 |

- 每 2 次下采样后接 **C2f**（CSP 风格残差块：两分支——主路 N 个 Bottleneck、旁路 1x1 卷积，最后 concat+1x1 融合），比旧版 C3 更轻更快。
- **SPPF**（Spatial Pyramid Pooling - Fast）：3 个串行 5×5 最大池化 + concat，扩大感受野，几乎不增加计算量。
- 输出 `P3/P4/P5` 三个尺度特征图供 neck 使用。


## 3. Neck (FPN+PAN 特征金字塔) 解读

```
 9 (P5/32, 1024ch)
 │ nn.Upsample ×2
 ├─ Concat(层6 P4) → C2f(512)   # 12: 自顶向下融合 P5+P4
 │ nn.Upsample ×2
 ├─ Concat(层4 P3) → C2f(256)   # 15 (P3/8)   ← 小目标
 │ Conv(3x3, s=2)
 ├─ Concat(层12) → C2f(512)     # 18 (P4/16)  ← 中目标
 │ Conv(3x3, s=2)
 ├─ Concat(层9)  → C2f(1024)    # 21 (P5/32)  ← 大目标
 │
 └─ [[15, 18, 21], 1, Pose, [nc, kpt_shape]]   # 关键: 三个尺度进 Pose 头
```

- 与检测版完全相同的 **FPN(自顶向下) + PAN(自底向上)** 结构：高层语义信息往下传（提升小目标），底层细节往上走（提升大目标）。
- 三个输出尺度 `P3/8, P4/16, P5/32`（相对 640 输入，特征图 80/40/20），覆盖不同大小的目标。
- 最后一行 `Pose` 头一次接收 3 个尺度特征图，各自独立预测框/类/关键点，再按 anchor 拼接。


## 5. Pose 检测头源码解读 (head.py)

```python
class Pose(Detect):                       # 继承检测头
    def __init__(self, nc, kpt_shape=(17, 3), reg_max=16, end2end=False, ch=()):
        super().__init__(nc, reg_max, end2end, ch)   # 复用 cv2(框) + cv3(类)
        self.kpt_shape = kpt_shape        # (17, 3) = 17 个点 × 3 维
        self.nk = kpt_shape[0] * kpt_shape[1]        # 17*3 = 51
        c4 = max(ch[0] // 4, self.nk)     # 关键点分支中间通道
        self.cv4 = nn.ModuleList(         # 每个尺度一个关键点分支
            nn.Sequential(
                Conv(x, c4, 3),           # 3x3 卷积 (骨干通道 -> c4)
                Conv(c4, c4, 3),          # 3x3 卷积
                nn.Conv2d(c4, self.nk, 1) # 1x1 -> 51 个通道 (17点×3维)
            ) for x in ch)
```

**要点**
- `Pose` 继承 `Detect`，**框分支 cv2 + 类分支 cv3 完全不变**，只新增 `cv4` 关键点分支。
- 关键点分支结构 = `Conv(3x3) → Conv(3x3) → Conv2d(1x1)`，输出通道数 = `nk = 17×3 = 51`。
- 每个特征图尺度（P3/P4/P5）各有一个 `cv4`，输出形状 `(bs, 51, H, W)`。
- `forward_head` 中把 3 个尺度的关键点特征 `view(bs, nk, -1)` 后沿 anchor 维度拼接，得到 `(bs, nk, 总anchor数)`。
- `fuse()` 在推理优化时会丢弃 one2many 分支；`one2many/one2one` 属性分别暴露框/类/关键点三个头（end2end 模式支持）。


## 6. 关键点解码原理 (kpts_decode)

训练/推理时，`Pose._inference()` 会把 框+类别 结果与 `kpts_decode()` 的结果拼接：

```python
preds = super()._inference(x)                          # [box(4) + cls(nc)]
kpts = self.kpts_decode(x["kpts"])                     # [nk] 关键点
return torch.cat([preds, kpts], dim=1)                 # 4 + nc + nk
```

**解码公式**（与框解码同一套 anchor 机制）：

$$x_{kpt} = \big(2\cdot p_x + (a_x - 0.5)\big) \cdot s$$
$$y_{kpt} = \big(2\cdot p_y + (a_y - 0.5)\big) \cdot s$$
$$v_{kpt} = \sigma(p_v) \quad (\text{若 } kpt\_shape[1]=3)$$

- 网络输出的是相对 anchor 的**偏移量**，`2*p + (a-0.5)` 先换算到该尺度特征图坐标，再乘 `stride` 还原到原图。
- 第 3 维 `visible` 用 **sigmoid** 归一化到 (0,1)，表示该关键点可见/遮挡概率（COCO 里 0=遮挡、1=可见、2=未标注）。
- 解码后与框/类拼接，再统一做 NMS 等后处理，最终每行输出格式：
  `[x1, y1, x2, y2, conf, class, kpt1_x, kpt1_y, kpt1_v, ..., kpt17_v]`

**前向/后处理流程 (Pose)**
1. `forward_head`：3 个尺度分别过 box/cls/pose 三个分支，把特征图拉平成 anchor 维度拼接。
2. 训练时直接返回原始预测（损失函数内部做正负样本匹配）；推理时 `_inference` 解码。
3. `postprocess`：按得分取 Top-K（`max_det`），输出 `[box, conf, class, kpts]`。


## 7. 与检测版 YOLOv8 的区别 & 在本项目(骨龄)中的应用

**vs 普通 YOLOv8 (yolov8.yaml)**
| 项目 | YOLOv8 | YOLOv8-pose |
|---|---|---|
| 输出 | 框 + 类别 | 框 + 类别 + 关键点 |
| 头部 | Detect (cv2/cv3) | Pose (cv2/cv3/cv4) |
| 每 anchor 输出数 | $4 + nc$ | $4 + nc + nk$ |
| 损失 | Box+Cls | Box+Cls+**Keypoint** (OKS 加权) |

Pose 头**完全复用** Detect 的框/类分支，只是**额外新增**一条关键点分支 `cv4`，所以主干和 neck 与检测版一模一样，代价极小。

**在骨龄项目中的潜在用途**
- 关键点 = 解剖学关节中心（如桡骨远端、各指骨关节）→ 比"纯检测框"更精准地**定位关节中心**，再按关节中心**裁剪**出 9 类关节图（替代现在的框裁剪）。
- 关键点可见性 (visible) 可作为骨发育程度的弱先验。
- 注意：本项目手骨是 7 类检测，若改用 pose 需要**重新标注关键点**（17 点对应 COCO 人体骨架，手骨数据集需要自定义 kpt_shape），标注成本较高，目前检测框方案已够用。


In [2]:
# 从 yaml 构建模型并查看结构 (yolov8-pose.yaml 默认 nc=1, 17 个关键点)
from ultralytics import YOLO

model = YOLO(r"d:\project\step1\week12\ultralytics-8.4.113\ultralytics\cfg\models\v8\yolov8-pose.yaml")
model.info()          # 层数 / 参数量 / GFLOPs

head = model.model.model[-1]   # PoseModel -> 内部 nn.Sequential -> 最后一个模块是 Pose 头
print("\nPose 检测头模块: ", type(head).__name__)
print("kpt_shape =", head.kpt_shape, " nk =", head.nk)
print("检测头输入通道 ch =", head.cv4[0][0].conv.in_channels)

WARNING no model scale passed. Assuming scale='n'.
YOLOv8-pose summary: 145 layers, 3,295,470 parameters, 3,295,454 gradients, 9.3 GFLOPs

Pose 检测头模块:  Pose
kpt_shape = [17, 3]  nk = 51
检测头输入通道 ch = 64


# 用 YOLOv8-pose 训练 luosi-keypoint 数据集

## 数据集说明 (luosi-keypoint)
- 位置: `luosi-keypoint/labelme_output/` — 螺丝目标检测 + 6 个关键点标注
- 每张图: 1 个 `luosi` 矩形框 + 6 个 point 关键点 (标签 `0`~`5`, 螺丝的 6 个顶点)
- 规模: 644 个 JSON + 647 张 JPG (部分 JSON/图片缺失, 代码会跳过)
- 目标格式 (YOLO-pose): `class_id x_c y_c w h x0 y0 v0 ... x5 y5 v5`, kpt_shape = [6, 3]

## 流程 (两个代码单元格)
1. **数据准备**: LabelMe JSON → YOLO-pose TXT, 8:2 随机划分 train/val, 生成 `dataset.yaml`
   - 原脚本 `1-labelme2yoloposetxt.py` / `2-split_dataset.py` 硬编码了 `E:/zs_kejian/...` 路径且换行写错(`"/n"`), 这里用本地路径重写
2. **训练**: `yolov8n-pose` 迁移学习
   - 预训练权重 `kpt_shape=17` 与本数据集 `6` 不一致时, 训练器自动重建 Pose 头 (主干保留预训练权重)

In [1]:
# ============================================================
# 1. 数据准备: LabelMe JSON → YOLO-pose TXT + 8:2 划分 + dataset.yaml
#    注意: luosi-keypoint/ 里自带的两个脚本写的是 E:/zs_kejian/... 路径
#    本机数据在工作区 d:\project\step1\week12\luosi-keypoint\labelme_output
#    这里用本地路径重写一版 (含原脚本漏写的 "\n" 换行修复)
# ============================================================
import json, shutil, random
from pathlib import Path

SRC = Path(r"d:\project\step1\week12\luosi-keypoint\labelme_output")
DST = Path(r"d:\project\step1\week12\luosi-keypoint\yolo_dataset")
TRAIN_RATIO, SEED = 0.8, 42

def parse_labelme(json_path: Path):
    """解析单个 LabelMe JSON → YOLO-pose 行字符串, 失败返回 None
    格式: 0 x_c y_c w h x0 y0 v0 ... x5 y5 v5 (6 关键点, 各带 visible=1.0)"""
    data = json.loads(json_path.read_text(encoding="utf-8"))
    w, h = data["imageWidth"], data["imageHeight"]
    if w <= 0 or h <= 0:
        return None
    # Step 1: luosi 矩形框 → 归一化中心/宽高
    bbox = None
    for s in data.get("shapes", []):
        if s.get("shape_type") == "rectangle" and s.get("label") == "luosi":
            (x1, y1), (x2, y2) = s["points"]
            x1, x2 = min(x1, x2), max(x1, x2)
            y1, y2 = min(y1, y2), max(y1, y2)
            bbox = ((x1 + x2) / 2 / w, (y1 + y2) / 2 / h, (x2 - x1) / w, (y2 - y1) / h)
            break
    if bbox is None:
        return None
    # Step 2: 6 个关键点 (label "0"~"5")
    kps = [0.0] * 18
    for s in data.get("shapes", []):
        if s.get("shape_type") == "point" and s.get("label") in "012345":
            x, y = s["points"][0]
            i = int(s["label"])
            kps[3 * i:3 * i + 3] = [x / w, y / h, 1.0]
    return "0 " + " ".join(f"{v:.6f}" for v in bbox) + " " + " ".join(f"{v:.6f}" for v in kps)

# 收集 (json + 对应图片) 样本
samples = []
for j in sorted(SRC.glob("*.json")):
    img = SRC / (j.stem + ".jpg")          # 如 123.jpg33.json → 123.jpg33.jpg
    if not img.exists():
        print(f"[SKIP] {j.name}: 无对应图片"); continue
    line = parse_labelme(j)
    if line is None:
        print(f"[SKIP] {j.name}: 无 luosi 矩形框"); continue
    samples.append((img, j.stem + ".txt", line))

# 8:2 随机划分
random.seed(SEED); random.shuffle(samples)
n = int(len(samples) * TRAIN_RATIO)
print(f"有效样本 {len(samples)} 张 → train {n} / val {len(samples) - n}")

# 写入 images/{train,val} + labels/{train,val}
for split, part in [("train", samples[:n]), ("val", samples[n:])]:
    for img, txt, line in part:
        (DST / "images" / split).mkdir(parents=True, exist_ok=True)
        (DST / "labels" / split).mkdir(parents=True, exist_ok=True)
        shutil.copy2(img, DST / "images" / split / img.name)
        (DST / "labels" / split / txt).write_text(line + "\n", encoding="utf-8")

# dataset.yaml
(DST / "dataset.yaml").write_text(
    f"""# YOLO-pose 数据集 (luosi-keypoint, 自动生成)
path: {DST.as_posix()}
train: images/train
val: images/val
kpt_shape: [6, 3]
flip_idx: [0, 1, 2, 3, 4, 5]
names:
  0: luosi
""", encoding="utf-8")
print("dataset.yaml →", DST / "dataset.yaml")

[SKIP] 147.jpg85.json: 无对应图片
[SKIP] 147.jpg86.json: 无对应图片
[SKIP] 2212.jpg234.json: 无对应图片
[SKIP] 2212.jpg235.json: 无对应图片
[SKIP] 863.jpg1917.json: 无对应图片
有效样本 639 张 → train 511 / val 128
dataset.yaml → d:\project\step1\week12\luosi-keypoint\yolo_dataset\dataset.yaml


In [ ]:
# ============================================================
# 2. 用 YOLOv8-pose 训练 luosi-keypoint 数据集
#    关键机制: 预训练 yolov8n-pose.pt 的 kpt_shape=17, 本数据集=6,
#              PoseTrainer.get_model() 会用数据集的 kpt_shape 覆盖并重建 Pose 头
#              (主干/neck 保留预训练权重, 只有关键点分支 cv4 重新初始化)
# ============================================================
from pathlib import Path
from ultralytics import YOLO

DATA = r"d:\project\step1\week12\luosi-keypoint\yolo_dataset\dataset.yaml"
WEIGHTS = Path(r"d:\project\step1\week12\yolov8n-pose.pt")

# 优先级: 本地已下载权重 → 自动下载 → 随机初始化
if WEIGHTS.exists():
    model = YOLO(str(WEIGHTS))
    print(f"[OK] 使用本地预训练权重: {WEIGHTS}")
else:
    try:
        model = YOLO("yolov8n-pose.pt")          # 自动下载预训练权重 (需联网, 较慢)
        print("[OK] 使用 yolov8n-pose.pt 预训练权重")
    except Exception as e:
        print(f"[WARN] 预训练权重不可用, 改用随机初始化: {e}")
        model = YOLO("yolov8n-pose.yaml")

# 训练 (训练器自动按 dataset.yaml 的 kpt_shape=[6,3] 重建 Pose 头)
results = model.train(
    data=DATA,
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,                        # RTX 3060 (0 = 第一块 GPU)
    patience=20,
    project=r"d:\project\step1\week12\luosi-keypoint\runs",
    name="luosi_pose",
    exist_ok=True,
)

# 在验证集上评估
metrics = model.val()
print(f"\nbox  mAP50={metrics.box.map50:.3f}  mAP50-95={metrics.box.map:.3f}")
print(f"pose mAP50={metrics.pose.map50:.3f}  mAP50-95={metrics.pose.map:.3f}")

# 可视化推理

训练好的模型在 `luosi-keypoint/runs/luosi_pose/weights/best.pt` (按 pose 指标选的最优权重)。

**best.pt 验证集指标 (训练时输出)**
- Box:  mAP50=0.995, mAP50-95=0.897
- Pose: mAP50=0.823, mAP50-95=0.657

下面用它在验证集上做推理:
- 每个螺丝输出 1 个检测框 + 6 个关键点 (螺丝 6 个顶点)
- 打印每张图的 框置信度 / 关键点像素坐标 / 可见性
- 标注图保存到 `runs/luosi_pose/visualize/`, 并展示前 3 张

In [3]:
# 保存标注图 + 打印每张的框/关键点信息
for r in results:
    name = Path(r.path).name
    if r.keypoints is None or len(r.keypoints) == 0:
        print(f"{name}: 未检测到目标"); continue
    box_conf = float(r.boxes.conf[0])
    kpts_xy = [[round(float(x), 1), round(float(y), 1)] for x, y in r.keypoints.xy[0]]   # (6,2) 像素坐标
    kpts_vis = [round(float(v), 2) for v in r.keypoints.conf[0]]                          # (6,) 可见性
    print(f"{name}: box_conf={box_conf:.3f}")
    print("  关键点 xy :", kpts_xy)
    print("  可见性    :", kpts_vis)
    cv2.imwrite(str(OUT / name), r.plot())     # r.plot() 返回带标注的 BGR 图

image0.jpg: box_conf=0.916
  关键点 xy : [[6.8, 14.1], [32.3, 12.0], [37.5, 32.1], [28.1, 60.2], [4.0, 56.6], [0.0, 27.1]]
  可见性    : [0.89, 0.86, 0.92, 0.94, 0.98, 0.97]
image1.jpg: box_conf=0.918
  关键点 xy : [[16.6, 10.4], [38.3, 12.9], [39.0, 40.1], [25.7, 62.8], [3.1, 60.2], [1.1, 28.4]]
  可见性    : [0.98, 0.96, 0.93, 0.84, 0.94, 0.97]
image2.jpg: box_conf=0.922
  关键点 xy : [[21.1, 3.0], [63.7, 0.8], [80.9, 34.6], [58.7, 65.5], [15.2, 60.9], [3.7, 30.6]]
  可见性    : [0.87, 0.87, 0.82, 0.63, 0.83, 0.9]
image3.jpg: box_conf=0.912
  关键点 xy : [[55.9, 25.7], [58.0, 29.1], [70.9, 51.0], [83.3, 86.1], [81.8, 83.8], [58.4, 59.1]]
  可见性    : [0.95, 0.97, 0.98, 0.99, 0.99, 0.99]
image4.jpg: box_conf=0.909
  关键点 xy : [[44.6, 13.4], [77.5, 13.3], [94.2, 39.8], [84.1, 79.1], [54.0, 75.5], [34.6, 45.3]]
  可见性    : [0.91, 0.94, 0.95, 0.94, 0.97, 0.98]
image5.jpg: box_conf=0.922
  关键点 xy : [[36.5, 23.0], [71.9, 27.9], [81.4, 58.0], [48.3, 82.1], [13.0, 77.2], [14.9, 41.2]]
  可见性    : [0.99, 0.97, 0.89, 0